# NBA Player Archetypes and All-NBA Prediction

## Data Collection and Cleaning

This notebook prepares NBA player-season data for:

1. Clustering NBA players into modern statistical archetypes.
2. Predicting whether a player is selected to an All-NBA team.

Each observation represents one player-season.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [3]:
df = pd.read_csv("../data/raw/nba_2024_25_raw.csv")

In [4]:
df.head()

,Rk,Player,WS,MP,G,Season,Age,Team,G.1,GS,AS,MP.1,FG,FGA,2P,2PA,3P,3PA,FT,FTA,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,FG%,2P%,3P%,FT%,TS%,eFG%,Pos
0,1,Shai Gilgeous-Alexander,16.7,2598,76,2024-25,26,OKC,76,76,1,2598,860,1656,697,1221,163,435,601,669,67,312,379,486,131,77,183,164,2484,0.519,0.571,0.375,0.898,0.637,0.569,G
1,2,Nikola Jokić,16.4,2571,70,2024-25,29,DEN,70,70,1,2571,786,1364,648,1033,138,331,361,451,200,692,892,716,127,45,230,160,2071,0.576,0.627,0.417,0.800,0.663,0.627,C-F
2,3,Ivica Zubac,11.7,2624,80,2024-25,27,LAC,80,80,0,2624,592,942,592,942,0,0,156,236,300,710,1010,214,55,90,127,168,1340,0.628,0.628,NaN,0.661,0.641,0.628,C
3,4,Jarrett Allen,11.6,2296,82,2024-25,26,CLE,82,82,0,2296,452,640,452,635,0,5,199,277,216,582,798,158,77,73,97,127,1103,0.706,0.712,0.000,0.718,0.724,0.706,C-F
4,5,Giannis Antetokounmpo,11.5,2289,67,2024-25,30,MIL,67,67,1,2289,793,1319,779,1256,14,63,436,707,147,651,798,433,58,78,206,155,2036,0.601,0.620,0.222,0.617,0.625,0.607,F-G


In [5]:
df.shape

(200, 36)

In [6]:
df.columns.tolist()

['Rk',
 'Player',
 'WS',
 'MP',
 'G',
 'Season',
 'Age',
 'Team',
 'G.1',
 'GS',
 'AS',
 'MP.1',
 'FG',
 'FGA',
 '2P',
 '2PA',
 '3P',
 '3PA',
 'FT',
 'FTA',
 'ORB',
 'DRB',
 'TRB',
 'AST',
 'STL',
 'BLK',
 'TOV',
 'PF',
 'PTS',
 'FG%',
 '2P%',
 '3P%',
 'FT%',
 'TS%',
 'eFG%',
 'Pos']

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 36 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Rk      200 non-null    int64  
 1   Player  200 non-null    str    
 2   WS      200 non-null    float64
 3   MP      200 non-null    int64  
 4   G       200 non-null    int64  
 5   Season  200 non-null    str    
 6   Age     200 non-null    int64  
 7   Team    200 non-null    str    
 8   G.1     200 non-null    int64  
 9   GS      200 non-null    int64  
 10  AS      200 non-null    int64  
 11  MP.1    200 non-null    int64  
 12  FG      200 non-null    int64  
 13  FGA     200 non-null    int64  
 14  2P      200 non-null    int64  
 15  2PA     200 non-null    int64  
 16  3P      200 non-null    int64  
 17  3PA     200 non-null    int64  
 18  FT      200 non-null    int64  
 19  FTA     200 non-null    int64  
 20  ORB     200 non-null    int64  
 21  DRB     200 non-null    int64  
 22  TRB     200 n

In [8]:
df.isna().sum()

Rk        0
Player    0
WS        0
MP        0
G         0
Season    0
Age       0
Team      0
G.1       0
GS        0
AS        0
MP.1      0
FG        0
FGA       0
2P        0
2PA       0
3P        0
3PA       0
FT        0
FTA       0
ORB       0
DRB       0
TRB       0
AST       0
STL       0
BLK       0
TOV       0
PF        0
PTS       0
FG%       0
2P%       0
3P%       5
FT%       0
TS%       0
eFG%      0
Pos       0
dtype: int64

In [9]:
print("G and G.1 identical:", df["G"].equals(df["G.1"]))
print("MP and MP.1 identical:", df["MP"].equals(df["MP.1"]))

G and G.1 identical: True
MP and MP.1 identical: True


In [10]:
df[["Player", "G", "G.1", "MP", "MP.1"]].head(10)

,Player,G,G.1,MP,MP.1
0,Shai Gilgeous-Alexander,76,76,2598,2598
1,Nikola Jokić,70,70,2571,2571
2,Ivica Zubac,80,80,2624,2624
3,Jarrett Allen,82,82,2296,2296
4,Giannis Antetokounmpo,67,67,2289,2289
5,Tyrese Haliburton,73,73,2451,2451
6,Rudy Gobert,72,72,2389,2389
7,Karl-Anthony Towns,72,72,2517,2517
8,Domantas Sabonis,70,70,2429,2429
9,Jayson Tatum,72,72,2624,2624


In [11]:
players = df.copy()

In [12]:
players = players.drop(columns=[
    "Rk",
    "G.1",
    "MP.1"
])

In [13]:
players.columns.tolist()

['Player',
 'WS',
 'MP',
 'G',
 'Season',
 'Age',
 'Team',
 'GS',
 'AS',
 'FG',
 'FGA',
 '2P',
 '2PA',
 '3P',
 '3PA',
 'FT',
 'FTA',
 'ORB',
 'DRB',
 'TRB',
 'AST',
 'STL',
 'BLK',
 'TOV',
 'PF',
 'PTS',
 'FG%',
 '2P%',
 '3P%',
 'FT%',
 'TS%',
 'eFG%',
 'Pos']

In [14]:
players.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 33 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Player  200 non-null    str    
 1   WS      200 non-null    float64
 2   MP      200 non-null    int64  
 3   G       200 non-null    int64  
 4   Season  200 non-null    str    
 5   Age     200 non-null    int64  
 6   Team    200 non-null    str    
 7   GS      200 non-null    int64  
 8   AS      200 non-null    int64  
 9   FG      200 non-null    int64  
 10  FGA     200 non-null    int64  
 11  2P      200 non-null    int64  
 12  2PA     200 non-null    int64  
 13  3P      200 non-null    int64  
 14  3PA     200 non-null    int64  
 15  FT      200 non-null    int64  
 16  FTA     200 non-null    int64  
 17  ORB     200 non-null    int64  
 18  DRB     200 non-null    int64  
 19  TRB     200 non-null    int64  
 20  AST     200 non-null    int64  
 21  STL     200 non-null    int64  
 22  BLK     200 n

In [15]:
players.isna().sum().sort_values(ascending=False)

3P%       5
Player    0
ORB       0
eFG%      0
TS%       0
FT%       0
2P%       0
FG%       0
PTS       0
PF        0
TOV       0
BLK       0
STL       0
AST       0
TRB       0
DRB       0
FTA       0
WS        0
FT        0
3PA       0
3P        0
2PA       0
2P        0
FGA       0
FG        0
AS        0
GS        0
Team      0
Age       0
Season    0
G         0
MP        0
Pos       0
dtype: int64

In [16]:
players["Player"].value_counts().head(20)

Player
Shai Gilgeous-Alexander    1
Nikola Jokić               1
Ivica Zubac                1
Jarrett Allen              1
Giannis Antetokounmpo      1
Tyrese Haliburton          1
Rudy Gobert                1
Karl-Anthony Towns         1
Domantas Sabonis           1
Jayson Tatum               1
Jalen Duren                1
Josh Hart                  1
Evan Mobley                1
Jimmy Butler               1
Payton Pritchard           1
Derrick White              1
Anthony Edwards            1
James Harden               1
Alperen Şengün             1
Jalen Brunson              1
Name: count, dtype: int64

In [17]:
players[players.duplicated(subset=["Player", "Season"], keep=False)][
    ["Player", "Season", "Team", "G", "MP", "PTS"]
].sort_values("Player")

,Player,Season,Team,G,MP,PTS


In [18]:
players["MPG"] = players["MP"] / players["G"]

players["PPG"] = players["PTS"] / players["G"]
players["RPG"] = players["TRB"] / players["G"]
players["APG"] = players["AST"] / players["G"]
players["SPG"] = players["STL"] / players["G"]
players["BPG"] = players["BLK"] / players["G"]
players["TOVPG"] = players["TOV"] / players["G"]

players["FGAPG"] = players["FGA"] / players["G"]
players["2PAPG"] = players["2PA"] / players["G"]
players["3PAPG"] = players["3PA"] / players["G"]
players["FTAPG"] = players["FTA"] / players["G"]

players["ORPG"] = players["ORB"] / players["G"]
players["DRPG"] = players["DRB"] / players["G"]

In [19]:
players[
    [
        "Player",
        "MPG",
        "PPG",
        "RPG",
        "APG",
        "SPG",
        "BPG",
        "3PAPG",
        "FTAPG"
    ]
].head(10)

,Player,MPG,PPG,RPG,APG,SPG,BPG,3PAPG,FTAPG
0,Shai Gilgeous-Alexander,34.184211,32.684211,4.986842,6.394737,1.723684,1.013158,5.723684,8.802632
1,Nikola Jokić,36.728571,29.585714,12.742857,10.228571,1.814286,0.642857,4.728571,6.442857
2,Ivica Zubac,32.800000,16.750000,12.625000,2.675000,0.687500,1.125000,0.000000,2.950000
3,Jarrett Allen,28.000000,13.451220,9.731707,1.926829,0.939024,0.890244,0.060976,3.378049
4,Giannis Antetokounmpo,34.164179,30.388060,11.910448,6.462687,0.865672,1.164179,0.940299,10.552239
5,Tyrese Haliburton,33.575342,18.616438,3.534247,9.219178,1.438356,0.671233,7.698630,3.041096
6,Rudy Gobert,33.180556,12.027778,10.902778,1.763889,0.777778,1.444444,0.000000,3.791667
7,Karl-Anthony Towns,34.958333,24.430556,12.777778,3.083333,0.986111,0.694444,4.694444,5.680556
8,Domantas Sabonis,34.700000,19.100000,13.885714,6.042857,0.685714,0.371429,2.228571,4.128571
9,Jayson Tatum,36.444444,26.833333,8.652778,5.986111,1.055556,0.527778,10.111111,6.111111


In [20]:
players.shape

(200, 46)

In [21]:
qualified_players = players[
    (players["G"] >= 20) &
    (players["MPG"] >= 15)
].copy()

In [22]:
qualified_players.shape

(196, 46)

In [23]:
print("Original players:", len(players))
print("Qualified players:", len(qualified_players))
print("Removed:", len(players) - len(qualified_players))

Original players: 200
Qualified players: 196
Removed: 4


In [24]:
all_nba = pd.read_csv("../data/raw/all_nba_selections.csv")

all_nba.head()

,Player,Season,All_NBA,All_NBA_Team
0,Nikola Jokic,2024-25,1,1
1,Giannis Antetokounmpo,2024-25,1,1
2,Jayson Tatum,2024-25,1,1
3,Shai Gilgeous-Alexander,2024-25,1,1
4,Donovan Mitchell,2024-25,1,1


In [25]:
qualified_players = qualified_players.merge(
    all_nba,
    on=["Player", "Season"],
    how="left"
)

In [26]:
qualified_players["All_NBA"] = (
    qualified_players["All_NBA"]
    .fillna(0)
    .astype(int)
)

qualified_players["All_NBA_Team"] = (
    qualified_players["All_NBA_Team"]
    .fillna(0)
    .astype(int)
)

In [27]:
qualified_players["All_NBA"].value_counts()

All_NBA
0    183
1     13
Name: count, dtype: int64

In [28]:
qualified_players[
    qualified_players["All_NBA"] == 1
][
    ["Player", "Season", "All_NBA_Team"]
].sort_values("All_NBA_Team")

,Player,Season,All_NBA_Team
0,Shai Gilgeous-Alexander,2024-25,1
4,Giannis Antetokounmpo,2024-25,1
9,Jayson Tatum,2024-25,1
28,Donovan Mitchell,2024-25,1
12,Evan Mobley,2024-25,2
16,Anthony Edwards,2024-25,2
19,Jalen Brunson,2024-25,2
23,Stephen Curry,2024-25,2
5,Tyrese Haliburton,2024-25,3
7,Karl-Anthony Towns,2024-25,3


In [29]:
missing_all_nba = all_nba[
    ~all_nba["Player"].isin(qualified_players["Player"])
]

missing_all_nba

,Player,Season,All_NBA,All_NBA_Team
0,Nikola Jokic,2024-25,1,1
6,Lebron James,2024-25,1,2


In [30]:
for player in missing_all_nba["Player"]:
    print("\nLooking for:", player)
    print(
        players[
            players["Player"]
            .str.contains(player.split()[-1], case=False, na=False)
        ][["Player", "Season", "G", "MPG"]]
    )


Looking for: Nikola Jokic
Empty DataFrame
Columns: [Player, Season, G, MPG]
Index: []

Looking for: Lebron James
          Player   Season   G        MPG
17  James Harden  2024-25  79  35.303797
25  LeBron James  2024-25  70  34.914286


In [31]:
qualified_players[
    qualified_players["All_NBA"] == 1
][
    ["Player", "Season", "G", "MPG", "All_NBA_Team"]
].sort_values(["All_NBA_Team", "Player"])

,Player,Season,G,MPG,All_NBA_Team
28,Donovan Mitchell,2024-25,71,31.436620,1
4,Giannis Antetokounmpo,2024-25,67,34.164179,1
9,Jayson Tatum,2024-25,72,36.444444,1
0,Shai Gilgeous-Alexander,2024-25,76,34.184211,1
16,Anthony Edwards,2024-25,79,36.341772,2
12,Evan Mobley,2024-25,71,30.521127,2
19,Jalen Brunson,2024-25,65,35.400000,2
23,Stephen Curry,2024-25,70,32.171429,2
50,Cade Cunningham,2024-25,70,35.028571,3
24,Jalen Williams,2024-25,69,32.420290,3


In [32]:
print("All-NBA file:", len(all_nba))
print("Successfully matched:", qualified_players["All_NBA"].sum())
print("Missing:", len(all_nba) - qualified_players["All_NBA"].sum())

All-NBA file: 15
Successfully matched: 13
Missing: 2


In [33]:
missing_all_nba

,Player,Season,All_NBA,All_NBA_Team
0,Nikola Jokic,2024-25,1,1
6,Lebron James,2024-25,1,2


In [34]:
players[
    players["Player"].str.contains("Jok", case=False, na=False)
][["Player", "Season", "G", "MPG"]]

,Player,Season,G,MPG
1,Nikola Jokić,2024-25,70,36.728571


In [35]:
players[
    players["Player"].str.contains("LeBron|Lebron|James", case=False, na=False)
][["Player", "Season", "G", "MPG"]]

,Player,Season,G,MPG
17,James Harden,2024-25,79,35.303797
25,LeBron James,2024-25,70,34.914286


In [36]:
all_nba = pd.read_csv("../data/raw/all_nba_selections.csv")

In [37]:
qualified_players = players[
    (players["G"] >= 20) &
    (players["MPG"] >= 15)
].copy()

In [38]:
qualified_players = qualified_players.merge(
    all_nba,
    on=["Player", "Season"],
    how="left"
)

In [39]:
qualified_players["All_NBA"] = (
    qualified_players["All_NBA"]
    .fillna(0)
    .astype(int)
)

qualified_players["All_NBA_Team"] = (
    qualified_players["All_NBA_Team"]
    .fillna(0)
    .astype(int)
)

In [40]:
qualified_players["All_NBA"].value_counts()

All_NBA
0    183
1     13
Name: count, dtype: int64

In [41]:
import unicodedata

def normalize_name(name):
    name = str(name)
    name = unicodedata.normalize("NFKD", name)
    name = "".join(
        c for c in name
        if not unicodedata.combining(c)
    )
    return name.lower().strip()

In [42]:
qualified_players["Player_Key"] = (
    qualified_players["Player"].apply(normalize_name)
)

all_nba["Player_Key"] = (
    all_nba["Player"].apply(normalize_name)
)

In [43]:
qualified_players = players[
    (players["G"] >= 20) &
    (players["MPG"] >= 15)
].copy()

qualified_players["Player_Key"] = (
    qualified_players["Player"].apply(normalize_name)
)

all_nba["Player_Key"] = (
    all_nba["Player"].apply(normalize_name)
)

In [44]:
qualified_players = qualified_players.merge(
    all_nba[
        ["Player_Key", "Season", "All_NBA", "All_NBA_Team"]
    ],
    on=["Player_Key", "Season"],
    how="left"
)

In [45]:
qualified_players["All_NBA"] = (
    qualified_players["All_NBA"]
    .fillna(0)
    .astype(int)
)

qualified_players["All_NBA_Team"] = (
    qualified_players["All_NBA_Team"]
    .fillna(0)
    .astype(int)
)

In [46]:
qualified_players["All_NBA"].value_counts()

All_NBA
0    181
1     15
Name: count, dtype: int64

In [47]:
qualified_players[
    qualified_players["All_NBA"] == 1
][
    ["Player", "Season", "All_NBA_Team"]
].sort_values(["All_NBA_Team", "Player"])

,Player,Season,All_NBA_Team
28,Donovan Mitchell,2024-25,1
4,Giannis Antetokounmpo,2024-25,1
9,Jayson Tatum,2024-25,1
1,Nikola Jokić,2024-25,1
0,Shai Gilgeous-Alexander,2024-25,1
16,Anthony Edwards,2024-25,2
12,Evan Mobley,2024-25,2
19,Jalen Brunson,2024-25,2
25,LeBron James,2024-25,2
23,Stephen Curry,2024-25,2


In [48]:
qualified_players.to_csv(
    "../data/processed/nba_2024_25_clean.csv",
    index=False
)

In [49]:
test = pd.read_csv("../data/processed/nba_2024_25_clean.csv")

print(test.shape)
print(test["All_NBA"].value_counts())

(196, 49)
All_NBA
0    181
1     15
Name: count, dtype: int64


In [50]:
qualified_players[
    qualified_players["All_NBA"] == 1
][
    ["Player", "Season", "G", "MPG", "PPG", "RPG", "APG", "All_NBA_Team"]
].sort_values("All_NBA_Team")

,Player,Season,G,MPG,PPG,RPG,APG,All_NBA_Team
0,Shai Gilgeous-Alexander,2024-25,76,34.184211,32.684211,4.986842,6.394737,1
1,Nikola Jokić,2024-25,70,36.728571,29.585714,12.742857,10.228571,1
4,Giannis Antetokounmpo,2024-25,67,34.164179,30.388060,11.910448,6.462687,1
9,Jayson Tatum,2024-25,72,36.444444,26.833333,8.652778,5.986111,1
28,Donovan Mitchell,2024-25,71,31.436620,23.957746,4.535211,4.985915,1
12,Evan Mobley,2024-25,71,30.521127,18.535211,9.281690,3.183099,2
16,Anthony Edwards,2024-25,79,36.341772,27.556962,5.696203,4.544304,2
19,Jalen Brunson,2024-25,65,35.400000,26.000000,2.876923,7.307692,2
23,Stephen Curry,2024-25,70,32.171429,24.542857,4.428571,6.014286,2
25,LeBron James,2024-25,70,34.914286,24.428571,7.800000,8.214286,2


Cleaning Multiple NBA Seasons

In [1]:
import pandas as pd
import numpy as np
import unicodedata

def normalize_name(name):
    name = str(name)
    name = unicodedata.normalize("NFKD", name)
    name = "".join(
        c for c in name
        if not unicodedata.combining(c)
    )
    return name.lower().strip()


def clean_season(filepath):
    df = pd.read_csv(filepath)

    # Preserve raw import, then work on a copy
    players = df.copy()

    # Remove unnecessary / duplicate Stathead columns
    columns_to_drop = ["Rk", "G.1", "MP.1"]

    players = players.drop(
        columns=[col for col in columns_to_drop if col in players.columns]
    )

    # Create per-game statistics
    players["MPG"] = players["MP"] / players["G"]
    players["PPG"] = players["PTS"] / players["G"]
    players["RPG"] = players["TRB"] / players["G"]
    players["APG"] = players["AST"] / players["G"]
    players["SPG"] = players["STL"] / players["G"]
    players["BPG"] = players["BLK"] / players["G"]
    players["TOVPG"] = players["TOV"] / players["G"]

    players["FGAPG"] = players["FGA"] / players["G"]
    players["2PAPG"] = players["2PA"] / players["G"]
    players["3PAPG"] = players["3PA"] / players["G"]
    players["FTAPG"] = players["FTA"] / players["G"]

    players["ORPG"] = players["ORB"] / players["G"]
    players["DRPG"] = players["DRB"] / players["G"]

    # Apply qualification rules
    players = players[
        (players["G"] >= 20) &
        (players["MPG"] >= 15)
    ].copy()

    # Create normalized merge key
    players["Player_Key"] = players["Player"].apply(normalize_name)

    return players

In [2]:
season_files = [
    "../data/raw/nba_2018_19_raw.csv",
    "../data/raw/nba_2019_20_raw.csv",
    "../data/raw/nba_2020_21_raw.csv",
    "../data/raw/nba_2021_22_raw.csv",
    "../data/raw/nba_2022_23_raw.csv",
    "../data/raw/nba_2023_24_raw.csv"
]

cleaned_seasons = []

for file in season_files:
    cleaned = clean_season(file)
    cleaned_seasons.append(cleaned)

    print(file, cleaned.shape)

../data/raw/nba_2018_19_raw.csv (192, 47)
../data/raw/nba_2019_20_raw.csv (197, 47)
../data/raw/nba_2020_21_raw.csv (198, 47)
../data/raw/nba_2021_22_raw.csv (199, 47)
../data/raw/nba_2022_23_raw.csv (195, 47)
../data/raw/nba_2023_24_raw.csv (196, 47)


In [3]:
historical_players = pd.concat(
    cleaned_seasons,
    ignore_index=True
)

historical_players.shape

(1177, 47)

In [4]:
players_2024_25 = pd.read_csv(
    "../data/processed/nba_2024_25_clean.csv"
)

In [5]:
players_2024_25["Player_Key"] = (
    players_2024_25["Player"].apply(normalize_name)
)

In [6]:
all_players = pd.concat(
    [
        historical_players,
        players_2024_25
    ],
    ignore_index=True
)

all_players.shape

(1373, 49)

In [7]:
for file in season_files:
    cleaned = clean_season(file)
    print(file, cleaned.shape)

../data/raw/nba_2018_19_raw.csv (192, 47)
../data/raw/nba_2019_20_raw.csv (197, 47)
../data/raw/nba_2020_21_raw.csv (198, 47)
../data/raw/nba_2021_22_raw.csv (199, 47)
../data/raw/nba_2022_23_raw.csv (195, 47)
../data/raw/nba_2023_24_raw.csv (196, 47)


In [8]:
all_nba = pd.read_csv("../data/raw/all_nba_selections.csv")

all_nba.shape

(105, 4)

In [9]:
all_nba["Player_Key"] = (
    all_nba["Player"].apply(normalize_name)
)

In [10]:
players_2024_25 = pd.read_csv(
    "../data/processed/nba_2024_25_clean.csv"
)

players_2024_25["Player_Key"] = (
    players_2024_25["Player"].apply(normalize_name)
)

all_players = pd.concat(
    [
        historical_players,
        players_2024_25
    ],
    ignore_index=True
)

print(all_players.shape)

(1373, 49)


In [11]:
all_players = all_players.drop(
    columns=["All_NBA", "All_NBA_Team"],
    errors="ignore"
)

In [12]:
all_players = all_players.merge(
    all_nba[
        [
            "Player_Key",
            "Season",
            "All_NBA",
            "All_NBA_Team"
        ]
    ],
    on=["Player_Key", "Season"],
    how="left"
)

In [13]:
all_players["All_NBA"] = (
    all_players["All_NBA"]
    .fillna(0)
    .astype(int)
)

all_players["All_NBA_Team"] = (
    all_players["All_NBA_Team"]
    .fillna(0)
    .astype(int)
)

In [14]:
all_players.groupby("Season")["All_NBA"].sum()

Season
2018-19    15
2019-20    15
2020-21    15
2021-22    15
2022-23    15
2023-24    15
2024-25    15
Name: All_NBA, dtype: int64

In [15]:
all_players["All_NBA"].value_counts()

All_NBA
0    1268
1     105
Name: count, dtype: int64

In [16]:
all_players.to_csv(
    "../data/processed/nba_player_seasons_clean.csv",
    index=False
)

In [17]:
master = pd.read_csv(
    "../data/processed/nba_player_seasons_clean.csv"
)

print(master.shape)
print(master["All_NBA"].value_counts())

(1373, 49)
All_NBA
0    1268
1     105
Name: count, dtype: int64


In [18]:
all_players.groupby("Season")["All_NBA"].sum()

Season
2018-19    15
2019-20    15
2020-21    15
2021-22    15
2022-23    15
2023-24    15
2024-25    15
Name: All_NBA, dtype: int64

In [19]:
print("Dataset shape:", all_players.shape)

print("\nAll-NBA distribution:")
print(all_players["All_NBA"].value_counts())

print("\nAll-NBA percentage:")
print(all_players["All_NBA"].value_counts(normalize=True) * 100)

Dataset shape: (1373, 49)

All-NBA distribution:
All_NBA
0    1268
1     105
Name: count, dtype: int64

All-NBA percentage:
All_NBA
0    92.352513
1     7.647487
Name: proportion, dtype: float64


In [20]:
duplicates = all_players[
    all_players.duplicated(
        subset=["Player_Key", "Season"],
        keep=False
    )
]

print("Duplicate player-season rows:", len(duplicates))

duplicates[
    ["Player", "Season", "Team", "G", "MPG", "PPG"]
].sort_values(["Season", "Player"])

Duplicate player-season rows: 0


,Player,Season,Team,G,MPG,PPG


In [21]:
all_players.isna().sum().sort_values(ascending=False)

3P%             37
Player           0
PTS              0
2P%              0
FT%              0
TS%              0
eFG%             0
Pos              0
MPG              0
PPG              0
RPG              0
APG              0
SPG              0
BPG              0
TOVPG            0
FGAPG            0
2PAPG            0
3PAPG            0
FTAPG            0
ORPG             0
DRPG             0
Player_Key       0
All_NBA          0
FG%              0
PF               0
WS               0
TOV              0
G                0
MP               0
Season           0
Age              0
Team             0
GS               0
AS               0
FG               0
FGA              0
2P               0
2PA              0
3P               0
3PA              0
FT               0
FTA              0
ORB              0
DRB              0
TRB              0
AST              0
STL              0
BLK              0
All_NBA_Team     0
dtype: int64

In [22]:
all_players.groupby("Season").size()

Season
2018-19    192
2019-20    197
2020-21    198
2021-22    199
2022-23    195
2023-24    196
2024-25    196
dtype: int64

In [23]:
all_players.columns.tolist()

['Player',
 'WS',
 'G',
 'MP',
 'Season',
 'Age',
 'Team',
 'GS',
 'AS',
 'FG',
 'FGA',
 '2P',
 '2PA',
 '3P',
 '3PA',
 'FT',
 'FTA',
 'ORB',
 'DRB',
 'TRB',
 'AST',
 'STL',
 'BLK',
 'TOV',
 'PF',
 'PTS',
 'FG%',
 '2P%',
 '3P%',
 'FT%',
 'TS%',
 'eFG%',
 'Pos',
 'MPG',
 'PPG',
 'RPG',
 'APG',
 'SPG',
 'BPG',
 'TOVPG',
 'FGAPG',
 '2PAPG',
 '3PAPG',
 'FTAPG',
 'ORPG',
 'DRPG',
 'Player_Key',
 'All_NBA',
 'All_NBA_Team']

In [24]:
all_players.to_csv(
    "../data/processed/nba_player_seasons_clean.csv",
    index=False
)

In [25]:
master = pd.read_csv(
    "../data/processed/nba_player_seasons_clean.csv"
)

print(master.shape)

print(
    master.groupby("Season")["All_NBA"].sum()
)

(1373, 49)
Season
2018-19    15
2019-20    15
2020-21    15
2021-22    15
2022-23    15
2023-24    15
2024-25    15
Name: All_NBA, dtype: int64


In [26]:
print("Duplicate player-season rows:", len(duplicates))

Duplicate player-season rows: 0


In [27]:
all_players.to_csv(
    "../data/processed/nba_player_seasons_clean.csv",
    index=False
)

master = pd.read_csv(
    "../data/processed/nba_player_seasons_clean.csv"
)

print(master.shape)
print(master.groupby("Season")["All_NBA"].sum())
print(master.duplicated(subset=["Player_Key", "Season"]).sum())

(1373, 49)
Season
2018-19    15
2019-20    15
2020-21    15
2021-22    15
2022-23    15
2023-24    15
2024-25    15
Name: All_NBA, dtype: int64
0
